In [95]:
import pandas as pd, json
from pathlib import Path

In [96]:
github_csv = pd.read_csv("projects.csv", dtype=str).fillna("")
github_csv["name_norm"] = github_csv["name"].str.lower().str.strip()

github_projects = [
    {
        "name": r["name"],
        "name_norm": r["name_norm"],
        "visibility": r.get("visibility", ""),
        "language": r.get("language", ""),
        "license": r.get("license", ""),
        "updated": r.get("updatedAt", "")
    }
    for _, r in github_csv.iterrows()
]

github_map = {p["name_norm"]: p for p in github_projects}


data = json.load(open("projects.json", "r", encoding="utf-8"))
prime = data.get("prime", {})

registry_projects = []
for category, items in prime.items():
    for it in items:
        ent = dict(it)
        ent["category"] = category
        ent["name_norm"] = ent.get("name", "").lower().strip()
        registry_projects.append(ent)

registry_map = {p["name_norm"]: p for p in registry_projects}

In [97]:
rows = []
registry_names = set(registry_map.keys())
github_names = set(github_map.keys())

for name_norm, reg in registry_map.items():
    gh = github_map.get(name_norm)  

    rows.append({
        "registry_name": reg.get("name"),
        "category": reg.get("category"),
        "status": reg.get("status", ""),

        "github_present": gh is not None,
        "github_name": gh["name"] if gh else "",
        "visibility": gh["visibility"] if gh else "",
        "language": gh["language"] if gh else "",
        "license": gh["license"] if gh else "",
        "updated": gh["updated"] if gh else ""
    })

df = (
    pd.DataFrame(rows)
    .sort_values(["category", "registry_name"])
    .reset_index(drop=True)
)

matched = df[df["github_present"]]
only_registry = df[~df["github_present"]]
only_github = sorted(github_names - registry_names)

print(f"Registry JSON total: {len(registry_projects)}")
print(f"GitHub CSV total: {len(github_projects)}")
print(f"Matched (in both): {len(matched)}")
print(f"Only in registry JSON: {len(only_registry)}")
print(f"Only in GitHub CSV: {len(only_github)}")

Registry JSON total: 56
GitHub CSV total: 56
Matched (in both): 56
Only in registry JSON: 0
Only in GitHub CSV: 0


In [98]:
if only_github:
    print("Projects in GitHub CSV not found in registry JSON (normalized names):")
    display(pd.DataFrame({"name_norm": only_github}))


if not only_registry.empty:
    print("Projects in registry JSON but missing in GitHub CSV:")
    display(only_registry[["registry_name", "category"]])

In [99]:
display(df)

,registry_name,category,status,github_present,github_name,visibility,language,license,updated
0,llmcolony,Chatbots,experimental,True,llmcolony,PUBLIC,,,2025-12-16 17:53:44
1,simtube,Chatbots,failed,True,simtube,PUBLIC,,,2025-04-23 20:22:55
2,telegramllmbot,Chatbots,successful,True,telegramllmbot,PUBLIC,,,2025-12-10 11:41:43
3,sparklineo,Data,-,True,sparklineo,PRIVATE,,,2026-02-10 09:46:48
4,echecs,Games,failed,True,echecs,PUBLIC,,,2025-03-23 23:11:24
5,games,Games,active,True,games,PUBLIC,,,2026-02-12 09:32:04
6,games.py,Games,active,True,games.py,PUBLIC,,,2026-02-12 11:07:04
7,games.rs,Games,-,True,games.rs,PUBLIC,,,2026-02-24 10:39:56
8,hunch,Games,merged,True,hunch,PUBLIC,,,2025-03-13 07:38:10
9,tictactoe,Games,-,True,tictactoe,PUBLIC,,,2025-03-24 02:13:19
